In [ ]:
import geopandas as gpd
from cartopy import crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

municipios = gpd.read_file('../temp/limites/SHP_ETRS89/recintos_municipales_inspire_peninbal_etrs89/recintos_municipales_inspire_peninbal_etrs89.shp')
municipio = municipios[(municipios.NAMEUNIT.str.startswith('Sangü')) | (municipios.NAMEUNIT.str.startswith('Aib'))]

municipio.to_crs(25830, inplace=True)
minx, miny, maxx, maxy = municipio.total_bounds
buffer = 1000

peg = gpd.read_file('PEG/propuesta_PEG.gpkg')

proj = ccrs.epsg('25830')

In [ ]:
comunidades = gpd.read_file('../temp/limites/SHP_ETRS89/recintos_provinciales_inspire_peninbal_etrs89/recintos_provinciales_inspire_peninbal_etrs89.shp')
navarra = comunidades[comunidades.NAMEUNIT == 'Navarra']

In [ ]:
import glob

masas_files = glob.glob('capas/Masas*.shp')

In [ ]:
masas = pd.concat([gpd.read_file(x) for x in masas_files])

In [ ]:
incendios1 = gpd.read_file('BRUTOS/HISTORICO/FOREST_Pol_HcoIncendio.shp')
incendios2 = gpd.read_file('BRUTOS/HISTORICO/FOREST_Pol_HcoIncendioA.shp')
incendios_totales = pd.concat([incendios1, incendios2])
incendios_buffer = incendios_totales[incendios_totales.intersects(municipio.dissolve().iloc[0].geometry.buffer(30000))]

incendios = incendios_buffer.sort_values('SUPQUEMADA', ascending=False).iloc[:10]

incendios.loc[incendios.FECHA == '2016', 'FECHA'] = '25/08/2016'
incendios.loc[incendios.FECHA == '1994', 'FECHA'] = '15/07/1994'
incendios.loc[incendios.FECHA == '2014', 'FECHA'] = '18/07/2014'
incendios.loc[incendios.FECHA == '2009', 'FECHA'] = '22/07/2009'

# CLIMOGRAMA

In [ ]:
from matplotlib import pyplot as plt

meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

def draw_climodiagrama(datos, a):
    datos = datos.loc[a]
    
    fig = plt.figure(figsize=(12,10))
    fig.suptitle('Precicpitación total: {:.2f}. Temperatura media: {:.2f}'.format(datos['prec'].sum(), datos['temp_media'].mean()), fontsize=20)
    plt.rc('axes', titlesize=20)
    plt.rc('axes', labelsize=20)
    plt.rc('xtick', labelsize=20)
    plt.rc('ytick', labelsize=20)

    ax1 = fig.add_subplot()
    ax2 = ax1.twinx()

    datos['prec'].plot(kind='bar', width=0.9, ax=ax1)
    datos['temp_media'].plot(color='salmon', ax=ax2, lw=2)
    datos['temp_max'].plot(color='red', ax=ax2, lw=2)
    datos['temp_min'].plot(color='orange', ax=ax2, lw=2)
    ax1.set_xticklabels(meses)

    ax2.set_ylim(0,  ax1.get_ylim()[1] / 2)

    ax1.yaxis.set_label_position("right")
    ax1.yaxis.tick_right()
    ax2.yaxis.set_label_position("left")
    ax2.yaxis.set_tick_params(labelcolor='red', color='red')
    ax1.yaxis.set_tick_params(labelcolor='C0', color='C0')

    ax2.yaxis.tick_left()
    ax1.spines['top'].set_visible(False)
    ax2.spines['top'].set_visible(False)

    ax2.set_ylabel("Temperatura Promedio (°C)", color='red')
    ax1.set_ylabel("Precipitación Promedio (mm)", color='C0')

    plt.show()
    fig.savefig('FINALES/CLIMOGRAMAS/climodiagrama_{}.png'.format(a), bbox_inches='tight', pad_inches=0)
    plt.close(fig)

In [ ]:
import clim_temp_media
import importlib
import os

importlib.reload(clim_temp_media)

c = clim_temp_media.Climograma('San Martín de Unx')
c.create_df()
c.draw_climodiagrama()

In [ ]:
datos = c.datos_anuales
for i in range(2005, 2023):
    draw_climodiagrama(datos, i)

In [ ]:
c.estacion

# GEOLOGÍA

In [ ]:
from pyproj import Transformer
import numpy as np
from osgeo import gdal

class VRF():
    def __init__(self, num):
        transformer = Transformer.from_crs("EPSG:4326", "EPSG:25830")

        ds = gdal.Open(f'../BRUTOS/GEOLOGIA/VRF_MAGNA50_{num}.jpg')
        gt = ds.GetGeoTransform()
        band1 = ds.GetRasterBand(1)
        band2 = ds.GetRasterBand(2)
        band3 = ds.GetRasterBand(3)

        b1 = band1.ReadAsArray()
        b2 = band2.ReadAsArray()
        b3 = band3.ReadAsArray()

        self.data = np.dstack((b1, b2, b3))
        ys, xs, c = detalle.shape
        ulx, xres, _, uly, _, yres = gt
        minx_, maxx_, maxy_, miny_ = [ulx, ulx+xres*xs, uly, uly+yres*ys]

        utm_minx, utm_maxy = transformer.transform(float(maxy_), float(minx_))
        utm_maxx, utm_miny = transformer.transform(float(miny_), float(maxx_))

        self.extent = [utm_minx, utm_maxx, utm_miny, utm_maxy]
        self.extent_g = [minx_, maxx_, miny_, maxy_]

In [ ]:
v1 = VRF(173)
v2 = VRF(206)

m = municipio.to_crs(4326)
p = ccrs.PlateCarree()

minx_, miny_, maxx_, maxy_ = m.total_bounds

fig = plt.figure(figsize=(16,12))
ax = plt.axes(projection=p)
ax.set_extent([minx_ - 0.01, maxx_ + 0.01, miny_ - 0.01, maxy_ + 0.01], crs=p)
m.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='b', linewidth=3)
ax.imshow(v1.data, extent=v1.extent_g)
ax.imshow(v2.data, extent=v2.extent_g)

In [ ]:
geo = gpd.read_file('../BRUTOS/GEOLOGIA/GEOLOG_Pol_Litologia.shp')
geo = geo[geo.intersects(municipio.iloc[0].geometry) == True]
geo.geometry = geo.apply(lambda x: x.geometry.intersection(municipio.iloc[0].geometry), axis=1)
display(geo)

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + 4*buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
geo.plot(ax=ax, column='NAME', legend=True, legend_kwds={'fontsize': 15}, alpha=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/geologia.png', bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + 2*buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
geo.plot(ax=ax, column='ERA', legend=True, legend_kwds={'fontsize': 15}, alpha=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/geologia_era.png', bbox_inches='tight')

# EDAFOLOGÍA

In [ ]:
edafo = gpd.read_file('BRUTOS/EDAFOLOGIA/EDAFOL_Pol_Suelos25m.shp')
edafo = edafo[edafo.intersects(municipio.dissolve().iloc[0].geometry) == True]
edafo.geometry = edafo.apply(lambda x: x.geometry.intersection(municipio.dissolve().iloc[0].geometry), axis=1)
display(edafo.keys())

In [ ]:
display(edafo.GEOMORF1.unique())

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + 14 * buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
edafo.plot(ax=ax, column='GEOMORF1', legend=True, legend_kwds={'fontsize': 13, 'loc':'lower right'}, alpha=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/edafologia.png', bbox_inches='tight')

In [ ]:
edafo[edafo.riesgo == 'No clasificado'].GEOMORF1.unique()

In [ ]:
import numpy as np

riesgo_alto = [
    'Escarpes y laderas de alta pendiente sobre margas, areniscas y limos', 
    'Laderas de erosión profundas sobre margas y areniscas', 
    'Laderas de erosión sobre margas y areniscas', 
    'Laderas esqueléticas entre terrazas', 
    'Lomas y laderas de erosión sobre margas con areniscas', 
    'Laderas de erosión bajo glacis y terrazas'
]

riesgo_medio = [
    'Glacis cementados provenientes de conglomerados y calizas', 
    'Terrazas medias cementadas del Aragón', 
    'Glacis provenientes de areniscas', 
    'Restos de terrazas y glacis no esqueléticos',
    'Glacis provenientes de conglomerados y de calizas',
    'Terrazas medias y glacis',
    'Glacis y distintos niveles de terraza del Irati, Areta y Salazar'
]

zona_transicion = [
    'Laderas de acumulación sobre margas grises', 
    'Laderas de acumulación y suaves vaguadas sobre arcillas, limos y areniscas', 
    'Laderas de acumulación sobre margas con areniscas',
    'Laderas de acumulación y suaves vaguadas bajo terrazas',
    'Vaguadas y laderas de acumulación sobre margas con areniscas',
    'Laderas de acumulación sobre margas y areniscas',
    'Terrazas medias del Aragón'
]

riesgo_bajo = [
    'Llanuras aluviales y terrazas más bajas', 
    'Zonas de inundación, cauces abandonados y barras de acreción lateral', 
    'Fondos de vaguada amplios', 'Fondo de barranco', 
    'Vaguadas sobre materiales variados', 
    'Vaguadas suaves en glacis',
    'Masas de agua (ríos, balsas...)'
]

zonas_antropicas = [
    'Casco urbano y carreteras', 
    'Canteras y graveras', 
    'Vertederos y escombreras'
]

condiciones = [
    edafo['GEOMORF1'].isin(riesgo_alto),
    edafo['GEOMORF1'].isin(riesgo_medio),
    edafo['GEOMORF1'].isin(zona_transicion),
    edafo['GEOMORF1'].isin(riesgo_bajo),
    edafo['GEOMORF1'].isin(zonas_antropicas)
]

# 2. Definimos las etiquetas correspondientes a cada condición
opciones = [
    'Riesgo Alto',
    'Riesgo Moderado-Alto',
    'Transición y Cultivo',
    'Riesgo Bajo / Barrera Natural',
    'Área Antrópica'
]

# 3. Creamos el nuevo campo 'riesgo' de forma vectorizada
edafo['riesgo'] = np.select(condiciones, opciones, default='No clasificado')

fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
edafo.plot(ax=ax, column='riesgo', legend=True, legend_kwds={'fontsize': 16, 'loc':'lower right'}, alpha=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/edafologia_riesgo.png', bbox_inches='tight')

# RÍOS

In [ ]:
import geopandas as gpd
from shapely.geometry import box

proj = ccrs.epsg('25830')

fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)
    
cuencas = gpd.read_file('BRUTOS/HIDROGRAFIA/HIDROG_Pol_CuencaHi.shp')
rios = gpd.read_file('BRUTOS/HIDROGRAFIA/HIDROG_Pol_RioPrincipal.shp')
rios.geometry = rios.geometry.buffer(100)

bbox_ze = box(minx, miny, maxx, maxy).buffer(1000)
print(maxx)

cuencas.plot(column='CUENCA', ax=ax, alpha=0.4, legend=True, linewidth=2, edgecolor='black', legend_kwds={'fontsize': 14})
municipio.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='red', linewidth=2)
rios.plot(ax=ax)
ax.add_geometries([bbox_ze], facecolor=(1,1,1,0), edgecolor='red', crs=proj)
plt.savefig('FINALES/IMAGENES/cuencas.png', bbox_inches='tight')

In [ ]:
import psycopg2
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
import numpy as np
import matplotlib.pyplot as plt
from cartopy import crs as ccrs

fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)
    
rios_ppales = gpd.read_file("./BRUTOS/HIDROGRAFIA/HIDROG_Pol_RioPrincipal.shp")
rios_secundarios = gpd.read_file("./BRUTOS/HIDROGRAFIA/HIDROG_Lin_CorrienteNatural.shp")

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='red', zorder=100, linewidth=4)

rios_ppales.plot(ax=ax, edgecolor='blue', linewidth=4, zorder=101)
rios_secundarios.plot(ax=ax, edgecolor='blue', linewidth=2, zorder=102)


# 1. Abrimos el raster original y calculamos la ventana de recorte (projWin)
with rasterio.open('./BRUTOS/merged_mdt.tif') as src:
    data_array = src.read(1).astype(float)
    raster_extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)


# Al usar matplotlib.pyplot.contour con 'extent', por lo general no necesitas 
# hacer np.flipud si las coordenadas de extensión están bien emparejadas, 
# pero mantenemos la lógica si tu flujo lo requería.
mdt_array = np.flipud(data_array)

ax.contour(
    data_array, 
    cmap='viridis', 
    levels=list(range(int(data_array[data_array > 0].min()), int(data_array.max()), 10)),
    extent=raster_extent,   # Posiciona las curvas en su lugar geográfico real
    origin='upper',         # Rasterio lee de arriba hacia abajo (norte a sur) igual que GDAL
    zorder=5
)

fig.savefig('FINALES/IMAGENES/hidrografia_curvas.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

# VEGETACIÓN

In [ ]:
ep = gpd.read_file('./BRUTOS/VEGETACION/BIODIV_Pol_RENA.shp')
ep = ep[ep.intersects(municipio.iloc[0].geometry) == True]

fig = plt.figure()
ax = plt.axes(projection=proj)

municipio.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='b', linewidth=2)
ep.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='g', linewidth=2)

In [ ]:
cultivos_orig = gpd.read_file('./BRUTOS/VEGETACION/OCUPAC_Pol_MCA_VE2021.shp')

cultivos = cultivos_orig[cultivos_orig.intersects(municipio.dissolve().iloc[0].geometry) == True]
cultivos['COBERTURAP'] = cultivos.apply(lambda x: x.COBERTURAP.capitalize() if x.COBERTURAP[1].isupper() else x.COBERTURAP, axis=1)
cultivos.geometry = cultivos.intersection(municipio.dissolve().iloc[0].geometry)
cultivos['SUP'] = cultivos.apply(lambda x: x.geometry.area / 10000, axis=1)

In [ ]:
forestal = cultivos[(cultivos.GRUPO == 'Forestal no arbolado') | (cultivos.GRUPO == 'Coníferas') | (cultivos.GRUPO == 'Coníferas/Frondosas') | (cultivos.GRUPO == 'Frondosas') ]
agraria = cultivos[(cultivos.GRUPO == 'Cultivos leñosos regadío') | (cultivos.GRUPO == 'Cultivos leñosos secano') | (cultivos.GRUPO == 'Cultivos herbáceos regadío') | (cultivos.GRUPO == 'Cultivos herbáceos secano') ]
industria = cultivos[(cultivos.COBERTURAP == 'POLÍGONO INDUSTRIAL ORDENADO') | (cultivos.COBERTURAP == 'POLÍGONO INDUSTRIAL SIN ORDENAR') | (cultivos.COBERTURAP == 'INDUSTRIA AISLADA')]

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + 6*buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
forestal.plot(ax=ax, column='COBERTURAP', legend=True, legend_kwds={'fontsize': 15, 'loc': 'lower right'}, alpha=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/forestal.png', bbox_inches='tight')

In [ ]:
forestal.keys()

In [ ]:
forestal.MOSAICO2.unique()

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + 4*buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
agraria.plot(ax=ax, column='COBERTURAP', legend=True, legend_kwds={'fontsize': 16, 'loc': 'lower right'}, alpha=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/agricola.png', bbox_inches='tight')

# GANADERÍA

In [ ]:
import glob

cierres_files = glob.glob('BRUTOS/PASTOS/*shp')

cierres = pd.concat([gpd.read_file(x) for x in cierres_files])

In [ ]:
ptos_sang = gpd.read_file('BRUTOS/PASTOS/PtosInteres_Sanguesa.shp')
ptos_aibar = gpd.read_file('BRUTOS/PASTOS/Aibar_PtosInteres.shp')
cierres_aibar = gpd.read_file('BRUTOS/PASTOS/Aibar_Cierres.shp')
corralizas_aibar = gpd.read_file('BRUTOS/PASTOS/Corralizas Aibar.shp')
corralizas_aibar

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
# ptos_sang.plot(ax=ax, column='PtosIntere', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
# ptos_aibar.plot(ax=ax, column='Tipo', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
# cierres_aibar.plot(ax=ax, column='Tipo', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
corralizas_aibar.plot(ax=ax, column='CORRALIZA', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/corralizas.png', bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
# ptos_sang.plot(ax=ax, column='PtosIntere', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
# ptos_aibar.plot(ax=ax, column='Tipo', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
cierres_aibar.plot(ax=ax, column='Tipo', legend=True, legend_kwds={'fontsize': 18}, linewidth=3, alpha=0.5, markersize=500)
# corralizas_aibar.plot(ax=ax, column='CORRALIZA', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/cierres.png', bbox_inches='tight')

In [ ]:
ptos_sang['Tipo'] = ptos_sang.PtosIntere

In [ ]:
fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

municipio.plot(ax=ax, facecolor=(1,0,1,0), edgecolor='black', linewidth=2)
# ptos_sang.plot(ax=ax, column='PtosIntere', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
pd.concat([ptos_sang, ptos_aibar]).plot(ax=ax, column='Tipo', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
# cierres_aibar.plot(ax=ax, column='Tipo', legend=True, legend_kwds={'fontsize': 18}, linewidth=3, alpha=0.5, markersize=500)
# corralizas_aibar.plot(ax=ax, column='CORRALIZA', legend=True, legend_kwds={'fontsize': 18}, edgecolor='black', linewidth=2, alpha=0.5, markersize=500)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/IMAGENES/pastos_puntos_interes.png', bbox_inches='tight')

# OROGRAFÍA

## ALTITUD

In [ ]:
import os
import geopandas as gpd
from cartopy import crs as ccrs
from matplotlib import pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import plotting_extent
from matplotlib.colors import LightSource

ls = LightSource(azdeg=315, altdeg=45) # Configuración estándar de la fuente de luz

# 1. Abrimos el archivo enmascarando los valores NoData
with rasterio.open('mdt_.tif') as ds:
    # masked=True transforma los NoData en NaN para que no rompan la escala
    data = ds.read(1, masked=True)  
    img_extent = plotting_extent(ds)

# Definimos la proyección cartopy usando el CRS del raster
proj = ccrs.epsg(25830)

fig = plt.figure(figsize=(16,12))
ax = plt.axes(projection=proj)

# Asegúrate de que municipio, minx, maxx, miny, maxy y buffer estén definidos antes
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

# 3. Dibujamos las capas utilizando el relieve real corregido
# Pintamos primero el relieve sombreado (Hillshade) básico
ax.imshow(ls.hillshade(data, vert_exag=1), extent=img_extent, origin='upper', cmap='gray', alpha=0.5)

# Pintamos encima el mapa de colores hipsométrico (Altitudes)
im = ax.imshow(data, extent=img_extent, origin='upper', cmap=plt.cm.gist_earth, alpha=0.6, 
               vmin=data.min(), vmax=data.max()) # Forzamos a matplotlib a ignorar los extremos rotos

# Dibujamos el vector del municipio
municipio.plot(ax=ax, facecolor=(1,0,0,0.1), edgecolor='black', linewidth=3)

# Vincular la barra de colores explícitamente a la capa de altitudes "im"
cb = fig.colorbar(im, shrink=.5, ax=ax, label='Altitud (m)')

os.makedirs('FINALES/OROGRAFIA', exist_ok=True)
plt.savefig('FINALES/OROGRAFIA/altitud.png', bbox_inches='tight')


## ORIENTACIÓN

In [ ]:
import os
import geopandas as gpd
from cartopy import crs as ccrs
from matplotlib import pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import plotting_extent  # Importación clave que añadiste
from matplotlib.colors import LightSource

ls = LightSource(azdeg=315, altdeg=45)

# 1. Abrimos el archivo enmascarando los valores NoData
with rasterio.open('./BRUTOS/aspect.tif') as ds:
    # Usamos masked=True para ignorar los valores vacíos fuera de Sangüesa/Aibar
    data = ds.read(1, masked=True)  
    # Usamos ds (el alias correcto del archivo abierto) para extraer el extent automáticamente
    img_extent = plotting_extent(ds)

# Definimos la proyección cartopy fija como hiciste en tu código modificado
proj = ccrs.epsg(25830)

fig = plt.figure(figsize=(16,12))
ax = plt.axes(projection=proj)

# Configuración de los límites del mapa (asegúrate de tener minx, maxx, miny, maxy y buffer declarados)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

# Dibujamos el vector del municipio
municipio.plot(ax=ax, facecolor=(1,0,0,0.1), edgecolor='black', linewidth=3)

# Pintamos el raster de orientaciones (aspect) utilizando el img_extent corregido
# Al usar origin='upper' de forma estándar con rasterio evitamos que el mapa salga invertido
im = ax.imshow(data, extent=img_extent, vmin=0, vmax=360, origin='upper', 
               cmap=plt.cm.twilight_shifted, alpha=0.8)

# Vinculamos la barra de colores cíclica (0-360º) explícitamente a nuestra capa "im"
cb = fig.colorbar(im, shrink=.5, ax=ax, label='Orientación (Grados Azimut)')

# Creamos la carpeta si no existe y guardamos el mapa final
os.makedirs('FINALES/OROGRAFIA', exist_ok=True)
plt.savefig('FINALES/OROGRAFIA/aspect.png', bbox_inches='tight')


In [ ]:
import os
import geopandas as gpd
from cartopy import crs as ccrs
from matplotlib import pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import plotting_extent
from matplotlib.colors import ListedColormap, BoundaryNorm

# 1. Abrimos el archivo de orientaciones enmascarando los valores NoData
with rasterio.open('./BRUTOS/aspect.tif') as ds:
    data = ds.read(1, masked=True)  
    img_extent = plotting_extent(ds)

# 2. Definimos los rangos para la leyenda discreta
# Rangos: 0-45 (Umbría), 45-135 (Transición), 135-225 (Solana Crítica), 225-315 (Transición), 315-360 (Umbría)
# Para simplificar la visualización en 3 clases, mapeamos los grados:
# Azul para Umbrías (Norte), Gris para Transiciones (E/O), Naranja/Rojo para Solanas (Sur)
cmap = ListedColormap(['#1f77b4', '#94979a', '#d62728', '#94979a', '#1f77b4'])
bounds = [0, 45, 135, 225, 315, 360]
norm = BoundaryNorm(bounds, cmap.N)

# Definimos la proyección cartopy fija (EPSG:25830)
proj = ccrs.epsg(25830)

fig = plt.figure(figsize=(16,12))
ax = plt.axes(projection=proj)

# Ajustamos la vista (requiere minx, maxx, miny, maxy y buffer)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

# Dibujamos el vector del municipio de fondo
municipio.plot(ax=ax, facecolor=(1,1,1,0.1), edgecolor='black', linewidth=3)

# Pintamos el raster reclasificado por umbrías y solanas
im = ax.imshow(data, extent=img_extent, origin='upper', alpha=0.8, 
               cmap=cmap, norm=norm)

# Configuramos una barra de colores personalizada y limpia
cb = fig.colorbar(im, shrink=.5, ax=ax, spacing='proportional', label='Exposición Solar (Laderas)')

# Colocamos los nombres en el centro de cada rango para que sea autoexplicativo
cb.set_ticks([22.5, 90, 180, 270, 337.5])
cb.set_ticklabels(['Umbría (N)', 'Transición (E)', 'Solana (S) - Crítica', 'Transición (O)', 'Umbría (N)'])

# Guardamos el mapa resultante
os.makedirs('FINALES/OROGRAFIA', exist_ok=True)
plt.savefig('FINALES/OROGRAFIA/solana_umbria.png', bbox_inches='tight')


## PENDIENTE

In [ ]:
import os
import geopandas as gpd
from cartopy import crs as ccrs
from matplotlib import pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import plotting_extent
from matplotlib.colors import LightSource
from matplotlib import colors

ls = LightSource(azdeg=315, altdeg=45)

# 1. Abrimos el archivo de pendientes enmascarando los valores NoData con Rasterio
with rasterio.open('./BRUTOS/slope.tif') as ds:
    # Usamos masked=True para ignorar píxeles vacíos fuera de los límites de estudio
    data = ds.read(1, masked=True)  
    # Extraemos el extent exacto usando el objeto nativo ds
    img_extent = plotting_extent(ds)


# Definimos la proyección cartopy fija (EPSG:25830)
proj = ccrs.epsg(25830)

fig = plt.figure(figsize=(16,12))
ax = plt.axes(projection=proj)

# Ajustamos la vista a los límites de tus municipios (requiere minx, maxx, miny, maxy y buffer)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

# Dibujamos el vector del municipio encima
municipio.plot(ax=ax, facecolor=(1,1,1,0.1), edgecolor='black', linewidth=3)

# Pintamos el raster aplicando tu segmentación de color y el img_extent corregido
# Usamos origin='upper' nativo de rasterio para evitar inversiones espaciales
im = ax.imshow(data, extent=img_extent, origin='upper', alpha=0.8)
 
# Vinculamos la barra de colores discretos explícitamente a nuestra capa "im"
cb = fig.colorbar(im, shrink=.5, ax=ax, spacing='proportional', label='Pendiente (%)')

# Creamos la carpeta de salida y guardamos el mapa definitivo
os.makedirs('FINALES/OROGRAFIA', exist_ok=True)
plt.savefig('FINALES/OROGRAFIA/slope_percent.png', bbox_inches='tight')


In [ ]:
import os
import geopandas as gpd
from cartopy import crs as ccrs
from matplotlib import pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import plotting_extent
from matplotlib.colors import LightSource
from matplotlib import colors

ls = LightSource(azdeg=315, altdeg=45)

# 1. Abrimos el archivo de pendientes enmascarando los valores NoData con Rasterio
with rasterio.open('./BRUTOS/slope.tif') as ds:
    # Usamos masked=True para ignorar píxeles vacíos fuera de los límites de estudio
    data = ds.read(1, masked=True)  
    # Extraemos el extent exacto usando el objeto nativo ds
    img_extent = plotting_extent(ds)

# Definimos tu mapa de colores discreto (Blanco < 30%, Rojo >= 30%)
cmap = colors.ListedColormap(['white', 'red'])
bounds = [0, 30, 100]
norm = colors.BoundaryNorm(bounds, cmap.N)

# Definimos la proyección cartopy fija (EPSG:25830)
proj = ccrs.epsg(25830)

fig = plt.figure(figsize=(16,12))
ax = plt.axes(projection=proj)

# Ajustamos la vista a los límites de tus municipios (requiere minx, maxx, miny, maxy y buffer)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

# Dibujamos el vector del municipio encima
municipio.plot(ax=ax, facecolor=(1,1,1,0.1), edgecolor='black', linewidth=3)

# Pintamos el raster aplicando tu segmentación de color y el img_extent corregido
# Usamos origin='upper' nativo de rasterio para evitar inversiones espaciales
im = ax.imshow(data, extent=img_extent, origin='upper', alpha=0.8, 
               cmap=cmap, norm=norm)

# Vinculamos la barra de colores discretos explícitamente a nuestra capa "im"
cb = fig.colorbar(im, shrink=.5, ax=ax, spacing='proportional', label='Pendiente (%)')
cb.set_ticks([15, 65])
cb.set_ticklabels(['< 30% (Baja/Media)', '>= 30% (Crítica)'])

# Creamos la carpeta de salida y guardamos el mapa definitivo
os.makedirs('FINALES/OROGRAFIA', exist_ok=True)
plt.savefig('FINALES/OROGRAFIA/slope_0_30.png', bbox_inches='tight')


# HISTÓRICO

In [ ]:
import os
import requests
import zipfile

if os.path.isdir('BRUTOS/HISTORICO'):
    pass
else:
    os.mkdir('BRUTOS/HISTORICO')
    
if  os.path.isfile('BRUTOS/HISTORICO/FOREST_Pol_HcoIncendio.shp'):
    pass
else:
    r = requests.get('https://idena.navarra.es/descargas/FOREST_Pol_HcoIncendio.zip')
    with open('BRUTOS/HISTORICO/incendios.zip', 'wb') as f:
      f.write(r.content)

    with zipfile.ZipFile('BRUTOS/HISTORICO/incendios.zip', 'r') as zip_ref:
        zip_ref.extractall('BRUTOS/HISTORICO/')

if  os.path.isfile('BRUTOS/HISTORICO/FOREST_Pol_HcoIncendioA.shp'):
    pass
else:
    r = requests.get('https://idena.navarra.es/descargas/FOREST_Pol_HcoIncendioA.zip')
    with open('BRUTOS/HISTORICO/incendios2.zip', 'wb') as f:
      f.write(r.content)

    with zipfile.ZipFile('BRUTOS/HISTORICO/incendios2.zip', 'r') as zip_ref:
        zip_ref.extractall('BRUTOS/HISTORICO/')

In [ ]:
resto_incendios = incendios_totales[~incendios_totales.intersects(municipio.dissolve().iloc[0].geometry.buffer(buffer_inc))]

In [ ]:
comunidades = gpd.read_file('BRUTOS/COMUNIDADES/comunidades.shp')
navarra = comunidades[comunidades['NAMEUNIT'] == 'Comunidad Foral de Navarra']

fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)

ax.set_extent([527946, 698200, 4633000, 4801100], crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
# ax.add_geometries(['geom'], facecolor=(0.1,0,0,0.2), edgecolor='red', linewidth=2, crs=proj)
resto_incendios.plot(ax=ax, facecolor='black')
incendios_buffer.plot(ax=ax, facecolor=(1,0,0,0.8), edgecolor='red')
navarra.plot(ax=ax, facecolor=(0.1,0,0,0.1), edgecolor='black', linewidth=4)
municipio.plot(ax=ax, facecolor=(0, 0, 0, 0), edgecolor='b', linewidth=3)
ax.add_geometries(
    [municipio.dissolve().iloc[0].geometry.buffer(buffer_inc)], 
    facecolor=(0.5, 0, 1, 0.1), edgecolor=(0.5, 0, 1), 
    linewidth=2, 
    crs=proj
)

legend_elements = [Patch(facecolor=(1,0,0,0.8), edgecolor='red', label='Incendios a menos de 30 km'),
                  Patch(facecolor=(0,0,0,0.8), edgecolor='black', label='Resto de incendios'),
                   Patch(facecolor=(0, 0, 1, 0.1), edgecolor='b', label='Límites de los municipios de Aibar y Sangüesa'),
                   Patch(facecolor=(0.5, 0, 1, 0.1), edgecolor=(0.5, 0, 1), label='Buffer de 30 km alrededor de los municipios')
                  ]
ax.legend(handles=legend_elements, fontsize=14)
 
fig.savefig('FINALES/IMAGENES/analisis_historico.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

In [ ]:
incendios_municipio = incendios_totales[incendios_totales.intersects(municipio.dissolve().iloc[0].geometry) == True]

fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.4)

incendios_municipio.plot(ax=ax, column='FECHAEXTIN', legend=True, legend_kwds={'loc': 'upper right', 'fontsize': 16}, alpha=0.4, edgecolor='red', linewidth=3)
municipio.plot(ax=ax, facecolor=(0.1,0,0,0.1), edgecolor='blue', linewidth=4)

# legend_elements = [Patch(facecolor=(1,0,0,0.8), edgecolor='red', label='20 incendios más grandes de Navarra'),
#                   Patch(facecolor=(0,0,0,0.8), edgecolor='black', label='Resto de incendios')]
# ax.legend(handles=legend_elements, fontsize=16)
 
fig.savefig('FINALES/IMAGENES/analisis_historico_municipio.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

## CIVIO

In [ ]:
# import psycopg2
# import pandas as pd
# import warnings

# warnings.filterwarnings('ignore')

# conn = psycopg2.connect('host=localhost dbname=navarra user=nano password=ventanuco')

# incendios_egif = gpd.read_postgis('''select deteccion, superficiearboladatotal, superficienoarboladatotal, d.causa, a.geom
#                 from incendios_navarra a, municipios b, causa_navarra c, causas d
#                 where st_contains(b.geom, a.geom) and b.cmunicipio = 217 and 
#                 a.numeroparte = c.numeroparte and d.codigo = c.idcausa
#                 order by deteccion''', conn)

# incendios_egif['suptotal'] = incendios_egif.apply(lambda x: (x.superficiearboladatotal + x.superficienoarboladatotal) * 800, axis=1)

## INCENDIOS CIVIO

civio = pd.read_csv('./BRUTOS/HISTORICO/fires-all.csv')

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Leer el archivo CSV de Civio
df = pd.read_csv("./BRUTOS/HISTORICO/fires-all.csv")

# 2. Filtrar solo los datos de la Comunidad Foral de Navarra (Código de provincia: 31)
# Nota: Si el dataset tuviera columna de texto directo, puedes ajustar a df['comunidad'] == 'Navarra'
df_navarra = df[df['idprovincia'] == 31].copy()

# 3. Eliminar filas donde la latitud o la longitud sean nulas o no válidas (0)
df_navarra = df_navarra.dropna(subset=['lat', 'lng'])
df_navarra = df_navarra[(df_navarra['lat'] != 0) & (df_navarra['lng'] != 0)]

# 4. Crear la columna de geometría (puntos) a partir de lng y lat
geometry = [Point(xy) for xy in zip(df_navarra['lng'], df_navarra['lat'])]

# 5. Convertir a GeoDataFrame asignando el sistema de coordenadas WGS84 (EPSG:4326)
# Es el sistema estándar para coordenadas de latitud/longitud decimales
gdf_navarra = gpd.GeoDataFrame(df_navarra, geometry=geometry, crs="EPSG:4326")

# Opcional: Mapear el diccionario de causas que vimos antes
diccionario_causas = {
    1: "Rayo",
    2: "Negligencia",
    3: "Intencionado",
    4: "Causa Desconocida",
    5: "Reproducción",
    6: "Accidental"
}
if 'causa' in gdf_navarra.columns:
    gdf_navarra['causa_texto'] = gdf_navarra['causa'].map(diccionario_causas)

# Verificar el resultado
print(f"Total incendios georreferenciados en Navarra: {len(gdf_navarra)}")
print(gdf_navarra[['id', 'lat', 'lng', 'causa_texto', 'geometry']].head())


In [ ]:
incendios_municipio = gdf_navarra.to_crs(25830)[gdf_navarra.to_crs(25830).intersects(municipio.dissolve().iloc[0].geometry)].copy()

In [ ]:
incendios_municipio.to_crs(25830).to_file('BRUTOS/HISTORICO/civio_municipio.gpkg')

In [ ]:
fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)

incendios_municipio.plot(ax=ax, column='causa_texto', legend=True, edgecolor='black', linewidth=1, markersize=incendios_municipio['superficie'] * 100)
municipio.plot(ax=ax, facecolor=(0.1,0,0,0.1), edgecolor='blue', linewidth=4)

# legend_elements = [Patch(facecolor=(1,0,0,0.8), edgecolor='red', label='20 incendios más grandes de Navarra'),
#                   Patch(facecolor=(0,0,0,0.8), edgecolor='black', label='Resto de incendios')]
# ax.legend(handles=legend_elements, fontsize=16)
 
fig.savefig('FINALES/IMAGENES/incendios_civio.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

## EGIF

In [ ]:
fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)

egif = gpd.read_file('BRUTOS/HISTORICO/egif.gpkg')
egif['superficie_total'] = egif.superficie_arbolada_ha + egif.superficie_no_arbolada_ha

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)

egif.plot(ax=ax, column='causa_general', legend=True, legend_kwds={'fontsize': 16}, edgecolor='black', linewidth=1, markersize=egif['superficie_total'] * 100)
municipio.plot(ax=ax, facecolor=(0.1,0,0,0.1), edgecolor='blue', linewidth=4)

# legend_elements = [Patch(facecolor=(1,0,0,0.8), edgecolor='red', label='20 incendios más grandes de Navarra'),
#                   Patch(facecolor=(0,0,0,0.8), edgecolor='black', label='Resto de incendios')]
# ax.legend(handles=legend_elements, fontsize=16)
 
fig.savefig('FINALES/IMAGENES/incendios_egif.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

# MEDIOS

## MEDIOS NAVARRA

In [ ]:
medios = gpd.read_file('./BRUTOS/MEDIOS/DOTACI_Sym_BomParques.shp')

minx_, miny_, maxx_, maxy_ = navarra.to_crs(25830).total_bounds

fig = plt.figure(figsize=(24,20))
ax = plt.axes(projection=proj)

ax.set_extent([minx_ - buffer, maxx_ + buffer, miny_ - buffer, maxy_ + buffer], crs=proj)

for n, value in medios.iterrows():
    if value.PARQUE == 'PAMPLONA-CORDOVILLA':
        ha = 'center'
        offsety = -4000
        offsetx = 0
    elif value.PARQUE ==  'MILUZE':
        ha = 'right'
        offsety = -1000
        offsetx = -2000
    else:
        ha = 'center'
        offsety = 2000
        offsetx = 0
    x = value.geometry.x + offsetx
    y = int(value.geometry.y) + offsety
    plt.text(x, y, value.PARQUE, horizontalalignment=ha, size=16)

municipio.plot(ax=ax, facecolor=(1,0,0,0.1), edgecolor='black', linewidth=2)
navarra.to_crs(25830).plot(ax=ax, facecolor=(0,0,1,0.02), edgecolor='black', linewidth=3)
medios.plot(ax=ax, markersize=200, column='tipo', legend='True', linewidth=2, edgecolor='black', legend_kwds={'fontsize':16})
comunidades.plot(ax=ax, facecolor=(0,0,1,0), edgecolor='black', linewidth=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/MEDIOS/medios_navarra.png', bbox_inches='tight')

## MEDIOS MINISTERIO

In [ ]:
medios_ministerio = gpd.read_file('./BRUTOS/MEDIOS/medios_ministerio.shp')
minx__, miny__, maxx__, maxy__ = medios_ministerio.total_bounds

_, _, _, maxy_ = navarra.to_crs(25830).total_bounds

fig = plt.figure(figsize=(18,26))
ax = plt.axes(projection=proj)
ax.set_extent([minx__ - 10000, maxx__ + 10000, miny__ - 10000, maxy_ + 10000], crs=proj)
navarra.plot(ax=ax, facecolor=(0,0,1,0.02), edgecolor='black', linewidth=3)
for n, value in medios_ministerio.iterrows():
    x = value.geometry.x
    y = int(value.geometry.y) + 3000
    plt.text(x, y, value.nombre, horizontalalignment='center', size=16)
medios_ministerio.plot(ax=ax, column='tipo', markersize=200, legend=True, linewidth=2, edgecolor='black', legend_kwds={'fontsize':16}, zorder=10)
navarra.to_crs(25830).plot(ax=ax, facecolor=(0,0,1,0.02), edgecolor='black', linewidth=3)
municipio.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='black', linewidth=2)
comunidades.plot(ax=ax, facecolor=(0,0,1,0), edgecolor='black', linewidth=0.5)
ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
plt.savefig('FINALES/MEDIOS/medios_ministerio.png', bbox_inches='tight')

# PISTAS FORESTALES

Las pistas forestales constituyen una estructura de vital importancia en la lucha contra incendios forestales. No sólo permiten la aproximación a las zonas de actuación para medios terrestres como autobombas o cuadrillas de tierra, sino que pueden constituir líneas de control en las que aplicar maniobras de fuego técnico. 

Se muestra mapa de 

In [ ]:
fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)
pistas = gpd.read_file('./BRUTOS/PISTAS/FOREST_Lin_PistasForP.shp')
pistas = pistas[pistas.intersects(municipio.dissolve().iloc[0].geometry)]
pistas['geometry'] = pistas.intersection(municipio.dissolve().iloc[0].geometry)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)

municipio.plot(ax=ax, facecolor=(0.1,0,0,0.1), edgecolor='blue', linewidth=4)
pistas.plot(ax=ax, edgecolor='black')
 
fig.savefig('FINALES/IMAGENES/pistas.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

# HIDRANTES

In [ ]:
hidrantes = gpd.read_file('../BRUTOS/HIDRANTES/hidrantes.shp')

minx_, miny_, maxx_, maxy_ = hidrantes.total_bounds
buffer_ = 250

In [ ]:
fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)
pistas = gpd.read_file('../BRUTOS/PISTAS/FOREST_Lin_PistasForP.shp')

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)

municipio.plot(ax=ax, facecolor=(0.1,0,0,0.1), edgecolor='blue', linewidth=4, label='San Martín de Unx')
hidrantes.plot(ax=ax, edgecolor='black', markersize=100, label='Hidrantes')

ax.legend(fontsize=18)

fig.savefig('FINALES/IMAGENES/hidrantes.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

In [ ]:
fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=proj)
pistas = gpd.read_file('../BRUTOS/PISTAS/FOREST_Lin_PistasForP.shp')

ax.set_extent([minx_ - buffer_, maxx_ + buffer_, miny_ - buffer_, maxy_ + buffer_], crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.9)

hidrantes.plot(ax=ax, edgecolor='black', markersize=150, label='Hidrantes')

# legend_elements = [Patch(facecolor=(1,0,0,0.8), edgecolor='red', label='20 incendios más grandes de Navarra'),
#                   Patch(facecolor=(0,0,0,0.8), edgecolor='black', label='Resto de incendios')]
ax.legend(fontsize=16)

fig.savefig('FINALES/IMAGENES/hidrantes_pueblo.png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

# PUNTOS DE AGUA

In [ ]:
ptos_agua = gpd.read_file('./BRUTOS/PUNTOS_AGUA/Puntos de agua actuales.shp')
ptos_agua = ptos_agua[ptos_agua.intersects(municipio.dissolve().iloc[0].geometry.buffer(2500)) == True]

In [ ]:
figure = plt.figure(figsize=(24, 20))
ax = plt.subplot(projection=proj)

minx__, miny__, maxx__, maxy__ = ptos_agua.total_bounds

minx_ = min(minx__, minx)
maxx_ = max(maxx__, maxx)
miny_ = min(miny__, miny)
maxy_ = max(maxy__, maxy)

ax.set_extent([minx_ - buffer, maxx_ + buffer, miny_ - buffer, maxy_ + buffer], crs=proj)
ptos_agua.plot(ax=ax, facecolor=(1,0,0,0.1), edgecolor='r', linewidth=4)
municipio.plot(facecolor=(1,0,0,0), edgecolor='black', linewidth=3, ax=ax)
for i, row in ptos_agua.iterrows():
    # if row.GEOM_AREA > 100:
        ax.add_geometries([row.geometry.buffer(2500)], linewidth=2, facecolor=(0,0,1,0.2), edgecolor='b', crs=proj)

ax.add_wms(wms='https://www.ign.es/wms-inspire/mapa-raster', layers=['mtn_rasterizado'], alpha=0.2)
plt.savefig(f'FINALES/IMAGENES/puntos_agua.png', bbox_inches='tight')

plt.show()

In [ ]:
from shapely.geometry import box
import os

for i, row in ptos_agua.iterrows():
    if os.path.isfile(f'FINALES/PUNTOS_AGUA/puntos_agua_{i}.png'):
        pass
    else:
        figure = plt.figure(figsize=(16, 10))
        ax = plt.subplot(projection=proj)

        buffer_ = 200

        minx_, miny_, maxx_, maxy_ = row.geometry.bounds

        ax.set_extent([minx_ - buffer_, maxx_ + buffer_, miny_ - buffer_, maxy_ + buffer_], crs=proj)

        ax.add_wms(wms='https://www.ign.es/wms-inspire/pnoa-ma', layers=['OI.OrthoimageCoverage'])
        plt.savefig(f'FINALES/PUNTOS_AGUA/puntos_agua_{i}.png', bbox_inches='tight')

        plt.show()
    
    if os.path.isfile(f'FINALES/PUNTOS_AGUA/localizacion_{i}.png'):
        pass
    else:
        figure = plt.figure(figsize=(16, 10))
        ax = plt.subplot(projection=proj)

        buffer_ = 200
        
        minx__, miny__, maxx__, maxy__ = row.geometry.bounds

        rect = box(minx__ - buffer_, miny__ - buffer_, maxx__ + buffer_, maxy__ + buffer_)

        minx_ = min(minx__, minx)
        maxx_ = max(maxx__, maxx)
        miny_ = min(miny__, miny)
        maxy_ = max(maxy__, maxy)

        ax.set_extent([minx_ - buffer, maxx_ + buffer, miny_ - buffer, maxy_ + buffer], crs=proj)
        ax.add_geometries([rect], crs=proj, facecolor=(1,0,0,0), linewidth=3, edgecolor='b')

        municipio.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='red', linewidth=3)
        ax.add_wms(wms='https://www.ign.es/wms-inspire/mapa-raster', layers=['mtn_rasterizado'], alpha=0.2)
        plt.savefig(f'FINALES/PUNTOS_AGUA/localizacion_{i}.png', bbox_inches='tight')

        plt.show()

# MODELOS DE COMBUSTIBLE

In [ ]:
mc = gpd.read_file('./BRUTOS/MC/FOREST_Pol_ModeCombus.shp')
mc = mc[mc.intersects(municipio.dissolve().iloc[0].geometry) == True]
mc.geometry = mc.apply(lambda x: x.geometry.intersection(municipio.dissolve().iloc[0].geometry), axis=1)
display(mc.keys())

In [ ]:
figure = plt.figure(figsize=(24, 20))
ax = plt.subplot(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)
municipio.plot(ax=ax)
mc.plot(ax=ax, column='IDCOMBUSTI', legend=True, legend_kwds={'fontsize': 16})

# ax.add_wms(wms='https://www.ign.es/wms-inspire/pnoa-ma', layers=['OI.OrthoimageCoverage'])

plt.savefig('FINALES/IMAGENES/mc_ant.png', bbox_inches='tight')

In [ ]:
figure = plt.figure(figsize=(24, 20))
ax = plt.subplot(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)
municipio.plot(ax=ax)
mc.plot(ax=ax, column='mc_nuevo', legend=True)

# ax.add_wms(wms='https://www.ign.es/wms-inspire/pnoa-ma', layers=['OI.OrthoimageCoverage'])

plt.savefig('FINALES/IMAGENES/mc_nuevo.png')

# LIDAR

In [ ]:
from owslib.wcs import WebCoverageService

wcs = WebCoverageService('https://idena.navarra.es/ogc/wcs', version='1.0.0')

out = wcs.getCoverage(identifier=wcs.contents['IDENA.WCS:ELEVAC_Ras_AltDom_10M_VE2017'].id,
                bbox=(minx, miny, maxx, maxy),
                format='image/tiff',
                crs='EPSG:25830', 
                resx=10,
                resy=10
)

with open('FINALES/LIDAR/arbolado.tif', 'wb') as f:
    f.write(out.read())

In [ ]:
wcs = WebCoverageService('https://idena.navarra.es/ogc/wcs', version='1.0.0')

out = wcs.getCoverage(identifier=wcs.contents['IDENA.WCS:ELEVAC_Ras_AltMed_10M_VE2017'].id,
                bbox=(minx, miny, maxx, maxy),
                format='image/tiff',
                crs='EPSG:25830', 
                resx=10,
                resy=10
)

with open('FINALES/LIDAR/matorral.tif', 'wb') as f:
    f.write(out.read())

In [ ]:
import matplotlib as mpl
from osgeo import gdal 

a = gdal.Open('FINALES/LIDAR/arbolado.tif')

band = a.GetRasterBand(1)
arbolado = band.ReadAsArray()

minx_, resx, j, maxy_, k, resy = a.GetGeoTransform()
dy, dx = arbolado.shape
img_extent = (minx_, minx_ + (dx * resx), maxy_ + (dy * resy), maxy_)

alpha = list(map(lambda x: list(map(lambda y: 0 if y == -9999 else 0.8, x)), arbolado))

fig = plt.figure(figsize=(16, 12))
ax = plt.subplot(projection=proj)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy +buffer], crs=proj)
municipio.plot(ax=ax, facecolor=(1,1,1,0), edgecolor='black', linewidth=4)
im = ax.imshow(arbolado, extent=img_extent, vmin=4, vmax=40, alpha=alpha)
colorbar = plt.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(4, 40), cmap='viridis'), shrink=0.7)
plt.savefig(f'FINALES/LIDAR/altura_arbolado.png', bbox_inches='tight')

In [ ]:
a = gdal.Open('FINALES/LIDAR/matorral.tif')
band = a.GetRasterBand(1)
arbolado = band.ReadAsArray()

minx_, resx, j, maxy_, k, resy = a.GetGeoTransform()
dy, dx = arbolado.shape
img_extent = (minx_, minx_ + (dx * resx), maxy_ + (dy * resy), maxy_)

alpha = list(map(lambda x: list(map(lambda y: 0 if y == -9999 else 0.8, x)), arbolado))

fig = plt.figure(figsize=(16, 12))
ax = plt.subplot(projection=proj)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy +buffer], crs=proj)
municipio.plot(ax=ax, facecolor=(1,1,1,0), edgecolor='black', linewidth=4)
im = ax.imshow(arbolado, extent=img_extent, vmin=0, vmax=4, alpha=alpha)
colorbar = plt.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 4), cmap='viridis'), shrink=0.7)
plt.savefig(f'FINALES/LIDAR/altura_matorral.png', bbox_inches='tight')

# SIMULACIONES

In [ ]:
from rasterio.plot import show

show(

In [ ]:
from cartopy import crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import owslib
import rasterio
from rasterio.plot import show
import geopandas as gpd
import fiona
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import glob

path = '../flammap/data'

for simulacion in range(2, 5):
    proj = ccrs.epsg('25830')

    paths = gpd.read_file(f'{path}/FINALES/IGNICION_SUR_{simulacion}/major_paths.shp')
    at_img = rasterio.open(f'{path}/FINALES/IGNICION_SUR_{simulacion}/at.tif')

    minx_paths, miny_paths, maxx_paths, maxy_paths = list(at_img.bounds)
        
    minx_, miny_, maxx_, maxy_ = municipio.dissolve().iloc[0].geometry.bounds

    minx = minx_
    maxx = max(maxx_paths, maxx_)
    miny = min(miny_paths, miny_)
    maxy = max(maxy_paths, maxy_)

    fig = plt.figure(figsize=(20, 16))
    buffer = 50
    ax = plt.axes(projection=proj)
    ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

    ax.add_wms(wms='https://www.ign.es/wms-inspire/mapa-raster', layers=['mtn_rasterizado'])

    show(at_img, ax=ax, cmap='Reds', zorder=99, alpha=0.7)
    show(at_img, ax=ax, cmap='Reds', contour=True, linewidths=2, zorder=100)
    municipio.plot(ax=ax, facecolor=(0,0,0,0), edgecolor='blue', linewidth=3)
    masas.plot(ax=ax, facecolor=(0,1,0,0.1), edgecolor='green', linewidth=2)

    for j in paths.geometry:
        ax.plot(j.xy[0], j.xy[1], linewidth=2, color='yellow', zorder=101)

    legend_elements = [Line2D([0], [0], color='y', lw=4, label='Carreras principales'),
                       Patch(facecolor=(0,0,0,0.2), edgecolor='blue', label='Límites municipales'),
                       Patch(facecolor=(0,1,0,0.2), edgecolor='green', label='Masas de ordenación'),
                       Patch(facecolor='orange', edgecolor='r',
                             label='Superficie incendio simulado'),
                       Line2D([0], [0], color='red', lw=4, label='Isocronas incendio simulado')]
    ax.legend(handles=legend_elements, prop={'size':16})
    plt.show()
    fig.savefig(f'FINALES/SIMULACIONES/simulacion_SUR_{simulacion}.png', bbox_inches='tight')
    plt.close(fig)

# PEGs

In [ ]:
legend_elements = [Patch(facecolor=(1,0,0,0), edgecolor='b', label='Límite Municipio'),
                   # Patch(facecolor=(0,1,0.9,0.9), edgecolor='green', label='PEG Zona de pastoreo'),
                   # Patch(facecolor=(1,0,0,0.2), edgecolor='red', label='PEG Faja auxiliar'),
                   # Patch(facecolor='brown', alpha=0.5, edgecolor='brown', label='Cierre ganadero'),
                   Patch(facecolor=(0.1, 0.9, 0.9, 0.5), edgecolor='cyan', label='PEG propuestos')    
                  # Patch(facecolor=(1,1,0,0.1), edgecolor='yellow', label='Monte comunal')
                  ]

figure = plt.figure(figsize=(24, 20))
ax = plt.subplot(projection=proj)

ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)
municipio.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='b', linewidth=3, zorder=10)
peg.plot(ax=ax, facecolor=(0,1, 0.9, 0.9), edgecolor='black', zorder=10)
# fajas.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='r', zorder=10)
# cierres.plot(ax=ax, facecolor='brown', alpha=0.4, edgecolor='brown', linewidth=2, zorder=10)
# monte.plot(ax=ax, facecolor=(1,1,0,0.1), edgecolor='yellow', linewidth=0.5, zorder=9)

ax.add_wms(wms='https://www.ign.es/wms-inspire/pnoa-ma', alpha=0.5, layers=['OI.OrthoimageCoverage'])
ax.legend(handles=legend_elements, fontsize=16)

plt.savefig('FINALES/IMAGENES/peg.png', bbox_inches='tight')
plt.close(figure)

## MAPAS 

In [ ]:
for i, row in peg.iterrows():
    legend_elements = [Patch(facecolor=(1,0,0,0), edgecolor='b', label='Límite Municipio'),
               Patch(facecolor=(0.1,0.9,0.3,0.2), edgecolor='red', label='PEG propuesto'),
              #  Patch(facecolor='brown', alpha=0.5, edgecolor='brown', label='Cierre ganadero'),
              # Patch(facecolor=(1,1,0,0.1), edgecolor='yellow', label='Monte comunal')
                      ]

    figure = plt.figure(figsize=(24, 20))
    ax = plt.subplot(projection=proj)

    minx_, miny_, maxx_, maxy_ = row.geometry.bounds
    
    ax.set_extent([minx_ - buffer, maxx_ + buffer, miny_ - buffer, maxy_ + buffer], crs=proj)
    municipio.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='b', linewidth=3, zorder=10)
    # peg.plot(ax=ax, facecolor=(0.1,0.9,0.3), edgecolor='g', zorder=10)
    ax.add_geometries([row.geometry], facecolor=(0.1, 0.9, 0.3, 0.2), edgecolor='g', linewidth=3, zorder=10, crs=proj)
    
    # fajas.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='r', zorder=10)
    # cierres.plot(ax=ax, facecolor='brown', alpha=0.4, edgecolor='brown', linewidth=2, zorder=10)
    # monte.plot(ax=ax, facecolor=(1,1,0,0.1), edgecolor='yellow', linewidth=0.5, zorder=9)
    
    ax.add_wms(wms='https://www.ign.es/wms-inspire/pnoa-ma', alpha=0.5, layers=['OI.OrthoimageCoverage'])
    ax.legend(handles=legend_elements, fontsize=16)

    plt.savefig(f'FINALES/PEG/peg_{row.id}.png', bbox_inches='tight')
    plt.close(figure)

## MAPAS TOPO

In [ ]:
import os
import rasterio
import matplotlib.pyplot as plt

# 1. Abrir el raster UNA SOLA VEZ fuera del bucle usando Rasterio
with rasterio.open('BRUTOS/merged_mdt.tif') as src:
    # Leemos la matriz de datos de la primera banda
    data_array = src.read(1).astype(float)
    # Rasterio nos da la extensión exacta (left, right, bottom, top) de forma nativa
    raster_extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

# Crear el directorio de salida si no existe
os.makedirs('FINALES/PEG', exist_ok=True)

for i, row in peg.iterrows():
    # Limpiamos figuras previas para evitar colapsar la memoria RAM
    fig, ax = plt.subplots(figsize=(24, 20), subplot_kw={'projection': proj})
    
    # Definir el encuadre con las variables corregidas (con guion bajo)
    minx_, miny_, maxx_, maxy_ = row.geometry.bounds
    
    # Configurar la ventana visual del mapa
    ax.set_extent([minx_ - buffer, maxx_ + buffer, miny_ - buffer, maxy_ + buffer], crs=proj)
    
    # Dibujar capas vectoriales
    municipio.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='b', linewidth=3, zorder=10)
    # fajas.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='r', zorder=10)
    # cierres.plot(ax=ax, facecolor='brown', alpha=0.4, edgecolor='brown', linewidth=2, zorder=10)
    # monte.plot(ax=ax, facecolor=(1,1,0,0.1), edgecolor='yellow', linewidth=0.5, zorder=9)
    ax.add_geometries([row.geometry], facecolor=(0.1, 0.9, 0.3, 0.2), edgecolor='g', linewidth=3, zorder=10, crs=proj)
    
    # 2. Dibujar las curvas de nivel usando los datos extraídos por Rasterio
    ax.contour(
        data_array, 
        cmap='viridis', 
        levels=list(range(int(data_array[data_array > 0].min()), int(data_array.max()), 10)),
        extent=raster_extent,   # Posiciona las curvas en su lugar geográfico real
        origin='upper',         # Rasterio lee de arriba hacia abajo (norte a sur) igual que GDAL
        zorder=5
    )
    
    ax.legend(handles=legend_elements, fontsize=16)
    
    # Guardar y cerrar la figura para liberar memoria
    plt.savefig(f'FINALES/PEG/peg_topo_{row.id}.png', bbox_inches='tight')
    plt.close(figure)


## MAPAS UBICACION

In [ ]:
from shapely.geometry import box

legend_elements = [Patch(facecolor=(1,0,0,0), edgecolor='b', label='Límite Municipio'),
           Patch(facecolor=(0.1,0.9,0.3,0.2), edgecolor='red', label='PEG propuesto'),
          #  Patch(facecolor='brown', alpha=0.5, edgecolor='brown', label='Cierre ganadero'),
          # Patch(facecolor=(1,1,0,0.1), edgecolor='yellow', label='Monte comunal')
                  ]
for i, row in peg.iterrows():
    figure = plt.figure(figsize=(24, 20))
    ax = plt.subplot(projection=proj)

    minx_, miny_, maxx_, maxy_ = row.geometry.bounds
    rect = box(minx_ - buffer, miny_ - buffer, maxx_ + buffer, maxy_ + buffer)
    
    ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)
    municipio.plot(ax=ax, facecolor=(1,0,0,0), edgecolor='b', linewidth=3, zorder=10)
    # peg.plot(ax=ax, facecolor=(0,1,0.9,0.3), edgecolor='g', zorder=10)
    # ax.add_geometries([row.geometry], facecolor=(0,1,0.9,0.3), edgecolor='g', linewidth=3, zorder=10, crs=proj)
    ax.add_geometries([rect], facecolor=(1,0,0,0), edgecolor='r', linewidth=3, zorder=10, crs=proj)
    
    # fajas.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='r', zorder=10)
    # cierres.plot(ax=ax, facecolor='brown', alpha=0.4, edgecolor='brown', linewidth=2, zorder=10)
    # monte.plot(ax=ax, facecolor=(1,1,0,0.1), edgecolor='yellow', linewidth=0.5, zorder=9)
    
    ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
    ax.legend(handles=legend_elements, fontsize=16)

    plt.savefig(f'FINALES/PEG/peg_ubicacion_{row.id}.png', bbox_inches='tight')
    plt.close(figure)

# RIESGO

In [ ]:
riesgos = gpd.read_file('../BRUTOS/RIESGO/riesgo_forestal.shp')

In [ ]:
fig = plt.figure(figsize=(24, 20))

ax = plt.axes(projection=proj)

riesgos.plot(ax=ax, column='riesgo', legend=True, edgecolor='black', linewidth=2, alpha=0.5, legend_kwds={'fontsize': 20}, cmap='autumn')
municipio.plot(ax=ax, facecolor=(0,0,1,0.6), edgecolor='black', zorder=10)

ax.add_wms(wms='https://www.ign.es/wms-inspire/ign-base', layers=['IGNBaseTodo'], alpha=0.6)
# ax.add_wms(wms='https://www.ign.es/wms-inspire/mapa-raster', layers=['mtn_rasterizado'], alpha=0.2)

plt.savefig('FINALES/IMAGENES/riesgos_pc.png'.format(nombre), bbox_inches='tight')

## PELIGRO

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

colors=[(0,1,0), (1,1,0), (1,0,0)]
cmap = LinearSegmentedColormap.from_list('my_list', colors, N=50)

In [ ]:
from shapely.geometry import box
ab = municipio.dissolve().iloc[0].geometry
cb = box(minx - buffer, miny - buffer, maxx + buffer, maxy + buffer)
mask = cb.difference(ab)

In [ ]:
import rasterio
from rasterio.plot import plotting_extent
import matplotlib.pyplot as plt
import matplotlib as mpl

with rasterio.open(f'./BRUTOS/RIESGO/mapa_peligro_verano_final_total_v2_final.tif') as src:
    a_ = src.read(1)
    img_extent = plotting_extent(src)

fig = plt.figure(figsize=(24, 20))
ax = plt.subplot(projection=proj)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy +buffer], crs=proj)

ax.add_geometries([mask], facecolor=(1,1,1,1), edgecolor='black', linewidth=4, crs=proj)

municipio.plot(ax=ax, facecolor=(1, 0, 0, 0), edgecolor='black', linewidth=2)
# Graficamos usando el extent calculado por rasterio
im = ax.imshow(a_, extent=img_extent, vmin=0, vmax=5, cmap=cmap)
# colorbar = plt.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 5), cmap=cmap), shrink=0.7)
plt.savefig(f'FINALES/IMAGENES/peligro.png', bbox_inches='tight')

## VULNERABILIDAD

In [ ]:
import rasterio
from rasterio.plot import plotting_extent
import matplotlib.pyplot as plt
import matplotlib as mpl

# Abrimos el archivo usando un administrador de contexto (with) para asegurar el cierre del archivo
with rasterio.open('./BRUTOS/RIESGO/vulnerabilidad_total_v2_final.tif') as src:
    # Leemos la banda 1 como un array de NumPy
    a_ = src.read(1)
    # rasterio calcula automáticamente el extent en formato (xmin, xmax, ymin, ymax) 
    # compatible con matplotlib / cartopy
    img_extent = plotting_extent(src)

# --- RESTO DE TU CÓDIGO DE VISUALIZACIÓN ---
fig = plt.figure(figsize=(24, 20))
ax = plt.subplot(projection=proj)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

ax.add_geometries([mask], facecolor=(1,1,1,1), edgecolor='black', linewidth=4, crs=proj)

municipio.plot(ax=ax, facecolor=(1, 0, 0, 0), edgecolor='black', linewidth=2)
# Graficamos usando el extent calculado por rasterio
im = ax.imshow(a_, extent=img_extent, vmin=0, vmax=5, cmap=cmap)
# colorbar = plt.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 5), cmap=cmap), shrink=0.7)

plt.savefig('FINALES/IMAGENES/vulnerabilidad.png', bbox_inches='tight')
plt.show()
plt.close(fig)


## RIESGO

In [ ]:
with rasterio.open('./BRUTOS/RIESGO/mapa_riesgo_verano_v2_final.tif') as src:
    a_ = src.read(1)
    img_extent = plotting_extent(src)

fig = plt.figure(figsize=(24, 20))
ax = plt.subplot(projection=proj)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy +buffer], crs=proj)

ax.add_geometries([mask], facecolor=(1,1,1,1), edgecolor='black', linewidth=4, crs=proj)

municipio.plot(ax=ax, facecolor=(1, 0, 0, 0), edgecolor='black', linewidth=2)
# Graficamos usando el extent calculado por rasterio
im = ax.imshow(a_, extent=img_extent, vmin=0, vmax=5, cmap=cmap)
# colorbar = plt.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 5), cmap=cmap), shrink=0.7)
plt.savefig(f'FINALES/IMAGENES/riesgo.png', bbox_inches='tight')

## INCENDIOS MAPAS INDIVIDUALES

In [ ]:
for i, row in incendios.iterrows():
    legend_elements = [Patch(facecolor=(1,0,0,0), edgecolor='red', label='Perímetro final incendio'),
              #  Patch(facecolor=(0.1,0.9,0.3,0.2), edgecolor='red', label='PEG propuesto'),
              # #  Patch(facecolor='brown', alpha=0.5, edgecolor='brown', label='Cierre ganadero'),
              # # Patch(facecolor=(1,1,0,0.1), edgecolor='yellow', label='Monte comunal')
                      ]

    figure = plt.figure(figsize=(24, 20))
    ax = plt.subplot(projection=proj)

    minx_, miny_, maxx_, maxy_ = row.geometry.bounds
    
    ax.set_extent([minx_ - buffer, maxx_ + buffer, miny_ - buffer, maxy_ + buffer], crs=proj)
    # peg.plot(ax=ax, facecolor=(0.1,0.9,0.3), edgecolor='g', zorder=10)
    ax.add_geometries([row.geometry], facecolor=(1, 0, 0, 0.2), edgecolor='red', linewidth=3, zorder=10, crs=proj)
    
    # fajas.plot(ax=ax, facecolor=(1,0,0,0.2), edgecolor='r', zorder=10)
    # cierres.plot(ax=ax, facecolor='brown', alpha=0.4, edgecolor='brown', linewidth=2, zorder=10)
    # monte.plot(ax=ax, facecolor=(1,1,0,0.1), edgecolor='yellow', linewidth=0.5, zorder=9)
    
    ax.add_wms(wms='https://www.ign.es/wms-inspire/mapa-raster', layers=['mtn_rasterizado'], alpha=0.2)
    # ax.add_wms(wms='https://www.ign.es/wms-inspire/pnoa-ma', alpha=0.5, layers=['OI.OrthoimageCoverage'])
    ax.legend(handles=legend_elements, fontsize=16)

    plt.savefig(f'BRUTOS/HISTORICO/MAPAS/mapa_{row.NUMPARTE}.png', bbox_inches='tight')
    plt.close(figure)

## WETTERZENTRALE

In [ ]:
import requests

incendios.loc[incendios.FECHA == '2016', 'FECHA'] = '25/08/2016'
incendios.loc[incendios.FECHA == '1994', 'FECHA'] = '15/07/1994'
incendios.loc[incendios.FECHA == '2014', 'FECHA'] = '18/07/2014'
incendios.loc[incendios.FECHA == '2009', 'FECHA'] = '22/07/2009'

PATH = 'BRUTOS/HISTORICO/WETTERZENTRALE'

for i, row in incendios.iterrows():
    dia, mes, year = row.FECHA.split('/')
    fecha = f'{year}{mes}{dia}'
    
    for iso in [{'num': 1, 'iso': '500'}, {'num': 2, 'iso': '850'}]:
        filename = f'{PATH}/isohipsas_{iso['iso']}_{fecha}.png'
        url = f'https://www.wetterzentrale.de/maps/archive/{year}/cfsr/CFSR_1_{fecha}18_{iso['num']}.png'

        r = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(r.content)

# r = requests.get('https://www.wetterzentrale.de/maps/archive/2026/cfsr/CFSR_1_2026071612_1.png')